# Publish the verified DeepSeek-Coder T4 LoRA adapter to Hugging Face

**Source:** `deepseek-coder-t4-smoke-adapter.zip` in your Google Drive (`MyDrive`).  
**Project:** [Birajkakadiya/deepseek-finetuning](https://github.com/Birajkakadiya/deepseek-finetuning)  
**Expected output:** a **new** model repository under your own Hugging Face account, containing **only the LoRA adapter**, tokenizer, model card, training metrics and loss plot.

**No GPU is required for publishing.** You can choose **Runtime → Change runtime type → CPU** to avoid consuming T4 hours.

Before publishing:
1. Log in or create a Hugging Face account: https://huggingface.co/join
2. Visit https://huggingface.co/settings/tokens and make a **write-capable access token** (use the least privileges available).
3. In Colab, select the **key (Secrets)** icon in the left sidebar. Add a secret named `HF_TOKEN`, paste your token, and allow this notebook to access the secret. **Never paste the token in a notebook cell or GitHub.**
4. Review the preview and model card. Publication is **off by default** until you explicitly change a switch.

The source base model has its own [DeepSeek Model License](https://huggingface.co/deepseek-ai/deepseek-coder-6.7b-instruct/blob/main/LICENSE), and [NVIDIA OpenCodeInstruct](https://huggingface.co/datasets/nvidia/OpenCodeInstruct) is CC BY 4.0. The generated model card references both. Review the license terms before public distribution.


## Step 1 — Mount your Google Drive and find the adapter ZIP

In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
ZIP_PATH = Path("/content/drive/MyDrive/deepseek-coder-t4-smoke-adapter.zip")
assert ZIP_PATH.is_file(), f"Adapter ZIP missing: {ZIP_PATH}. Adjust ZIP_PATH if you moved it."
print("Found adapter:", ZIP_PATH.name, "size:", round(ZIP_PATH.stat().st_size / 1024**2, 1), "MiB")

## Step 2 — Clone your GitHub project and install the publishing library

In [ ]:
import subprocess, sys
repo_dir = Path("/content/deepseek-finetuning")
repo_url = "https://github.com/Birajkakadiya/deepseek-finetuning.git"
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", repo_url, str(repo_dir)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "huggingface_hub"], check=True)
print("Publisher ready:", repo_dir / "publish_to_hub.py")

## Step 3 — Preview and validate the ZIP (no upload)

This performs the ZIP integrity check and confirms the training metadata matches the expected LoRA base model. It does not use your Hugging Face token.

In [ ]:
import os
REPO_NAME = "deepseek-coder-6.7b-opencodeinstruct-lora-t4-smoke"
VISIBILITY = "public"  # change to "private" if you prefer to review it before sharing
preview_cmd = [sys.executable, str(repo_dir / "publish_to_hub.py"),
               "--zip", str(ZIP_PATH), "--repo-name", REPO_NAME,
               "--visibility", VISIBILITY]
subprocess.run(preview_cmd, check=True)
print("Preview passed. No upload has happened.")

## Step 4 — Publish to Hugging Face (explicit opt-in)

Set `PUBLISH_TO_HUB = True` only when you are ready. **Public** means anyone can download your adapter weights and view the model card; **private** limits visibility to authorized accounts. The publisher **refuses to overwrite an existing model repository**.

A success message will print your actual model URL. No tokens will be printed or committed to GitHub.

In [ ]:
PUBLISH_TO_HUB = False  # Change to True only when ready to upload
if PUBLISH_TO_HUB:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("No HF_TOKEN found. Add it in Colab Secrets (key icon).")
    env = os.environ.copy()
    env["HF_TOKEN"] = token
    publish_cmd = preview_cmd + ["--publish"]
    try:
        subprocess.run(publish_cmd, env=env, check=True)
    finally:
        env.pop("HF_TOKEN", None)
        del token
    print("Open the PUBLISHED URL printed above to verify your model page.")
else:
    print("No upload performed. Set PUBLISH_TO_HUB=True to create and upload the model.")

## After publishing

1. Open the URL printed by the publisher, for example `https://huggingface.co/YOUR-USERNAME/deepseek-coder-6.7b-opencodeinstruct-lora-t4-smoke`.
2. Confirm that **Files and versions** includes `adapter_model.safetensors`, `adapter_config.json`, tokenizer files, `README.md`, and `training_metrics.json`.
3. Your Hub model card explains that the adapter is a **2-step proof-of-pipeline smoke test**, not a proven coding-performance improvement.
4. For a longer fine-tuning experiment, use [your T4 training notebook](https://colab.research.google.com/github/Birajkakadiya/deepseek-finetuning/blob/main/colab/DeepSeek_Coder_6_7B_T4_QLoRA.ipynb).

### Common errors

- **ZIP missing:** check Google Drive filename and adjust `ZIP_PATH`.
- **Token missing:** add the exact Colab Secret name `HF_TOKEN` and grant notebook access.
- **401 or 403:** verify the token's write permissions and account authorization.
- **Repository already exists:** choose a new `REPO_NAME` (existing model repos are deliberately not overwritten).
- **Upload interrupted:** check whether the repository exists and contains files; retry with a new name or remove the incomplete repository manually before reusing its name.
- **No T4 available:** this upload process uses CPU only; no GPU session is needed.
